In [1]:
from google.colab import drive
from pathlib import Path

# Mount Google Drive to the standard mountpoint
drive.mount("/content/drive")

# Access your specific workbench path
workbench_path = Path("/content/drive/MyDrive/Workbench")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
os.listdir(workbench_path)

['plan_2026_H1.csv',
 'patients.csv',
 'collections',
 'count_rule_comparison.csv',
 'rejects_step5.csv.gz',
 'decisions_log_auto.csv',
 'bookings_clean.csv.gz',
 'old_bookings_merged.csv',
 'bookings_merged.csv',
 'wb1.ipynb',
 'results']

In [3]:
import pandas as pd
df = pd.read_csv(workbench_path / "bookings_merged.csv")

/tmp/ipykernel_13292/1458564450.py:2: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(workbench_path / "bookings_merged.csv")


In [4]:
df.columns

Index(['booking_id', 'created_at', 'slot_at', 'city', 'clinic_id', 'channel',
       'source', 'patient_id', 'patient_name', 'test_code', 'test_price',
       'home_fee', 'source_file', 'system', 'month', 'pairs'],
      dtype='object')

In [5]:
df.head()

,booking_id,created_at,slot_at,city,clinic_id,channel,source,patient_id,patient_name,test_code,test_price,home_fee,source_file,system,month,pairs
0,BK-AHM-202501-000001,2025-01-08 03:16:00,2025-01-09 08:26:00,Ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T009,800,0,bookings_2025-01.csv.gz,legacy,2025-01,2025-01_Ahmedabad
1,BK-AHM-202501-000001,2025-01-08 03:16:00,2025-01-09 08:26:00,Ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T004,650,0,bookings_2025-01.csv.gz,legacy,2025-01,2025-01_Ahmedabad
2,BK-AHM-202501-000001,2025-01-08 03:16:00,2025-01-09 08:26:00,Ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T007,550,0,bookings_2025-01.csv.gz,legacy,2025-01,2025-01_Ahmedabad
3,BK-AHM-202501-000002,2025-01-03 05:18:00,2025-01-06 13:22:00,Ahmedabad,AHM-04,clinic,app,PAHM0031384,Nisha Iyer,T004,650,0,bookings_2025-01.csv.gz,legacy,2025-01,2025-01_Ahmedabad
4,BK-AHM-202501-000003,2025-01-27 10:46:00,2025-01-28 13:37:00,Ahmedabad,AHM-01,home,app,PAHM0027001,Sneha Menon,T003,450,150,bookings_2025-01.csv.gz,legacy,2025-01,2025-01_Ahmedabad


In [6]:
df['city'].unique()

array(['Ahmedabad', 'Bengaluru', 'Chennai', 'Hyderabad', 'Jaipur', 'Pune'],
      dtype=object)

In [7]:
known = df[df.test_code.notna()]
price_to_codes = known.groupby("test_price", observed=True).test_code.nunique()

missing = df[df.test_code.isna()]
m = missing.test_price.map(price_to_codes)

print("missing rows whose price maps to exactly 1 code:", (m == 1).sum())
print("maps to several codes:", (m > 1).sum())
print("price never seen with a code:", m.isna().sum())

missing rows whose price maps to exactly 1 code: 822
maps to several codes: 467
price never seen with a code: 0


#1 Count test both Halves

In [8]:

h25 = df[df.slot_at.between("2025-01-01", "2025-06-30 23:59:59")]
h26 = df[df.slot_at.between("2026-01-01", "2026-06-30 23:59:59")]
print("H1 2025:", len(h25), " H1 2026:", len(h26))
print("difference:", len(h26) - len(h25), " growth %:", round((len(h26)/len(h25)-1)*100, 1))
print(h25.groupby("city").size().to_frame("H1_2025").join(h26.groupby("city").size().to_frame("H1_2026")))

H1 2025: 435430  H1 2026: 457230
difference: 21800  growth %: 5.0
           H1_2025  H1_2026
city                       
Ahmedabad    76800    84914
Bengaluru    99811   107473
Chennai      80094    80676
Hyderabad    73133    78548
Jaipur       45071    40180
Pune         60521    65439


In [9]:
h25 = df[df.slot_at.between("2025-01-01", "2025-06-30 23:59:59")]
h26 = df[df.slot_at.between("2026-01-01", "2026-06-30 23:59:59")]

print("H1 2025:", len(h25), " H1 2026:", len(h26))
print(
    "difference:",
    len(h26) - len(h25),
    " growth %:",
    round((len(h26) / len(h25) - 1) * 100, 1),
)

# City-wise comparison table with Difference and Growth %
city_df = (
    h25.groupby("city")
    .size()
    .to_frame("H1_2025")
    .join(h26.groupby("city").size().to_frame("H1_2026"))
)
city_df["Difference"] = city_df["H1_2026"] - city_df["H1_2025"]
city_df["Growth_%"] = (
    (city_df["H1_2026"] / city_df["H1_2025"] - 1) * 100
).round(1)

print(city_df)

H1 2025: 435430  H1 2026: 457230
difference: 21800  growth %: 5.0
           H1_2025  H1_2026  Difference  Growth_%
city                                             
Ahmedabad    76800    84914        8114      10.6
Bengaluru    99811   107473        7662       7.7
Chennai      80094    80676         582       0.7
Hyderabad    73133    78548        5415       7.4
Jaipur       45071    40180       -4891     -10.9
Pune         60521    65439        4918       8.1


"""Step 3: compare our counts with the plan, city by city, month by month.
Run from repo root:  python src/step3_plan_check.py"""

In [10]:
df["slot_at"] = pd.to_datetime(df["slot_at"])

# Now you can safely use .dt accessors, for example:
df["month"] = df.slot_at.dt.strftime("%Y-%m")

In [11]:
# 1. Plan: wide -> long, same city spelling as our data
plan = pd.read_csv(workbench_path / "plan_2026_H1.csv").melt(id_vars=["city", "metric"], var_name="month", value_name="plan_2026")
plan["city"] = plan["city"].replace({"Bangalore": "Bengaluru"})
plan = plan[["city", "month", "plan_2026"]]

# 2. Our counts: one row per city per month. Rule for now: every booking-test row,
#    except nova records marked 'rescheduled' (the old copy of a moved visit).
if "status_at_export" in df.columns:
    superseded = df["status_at_export"].eq("rescheduled").fillna(False).astype(bool)
else:
    superseded = pd.Series(False, index=df.index)

cnt = df[~superseded].groupby(["city", "month"]).size().rename("tests").reset_index()

# 3. Last year's count for the same calendar month: shift 2025-MM to 2026-MM so we can join
ly = cnt[cnt.month.str.startswith("2025")].copy()
ly["month"] = ly.month.str.replace("2025", "2026", regex=False)
ly = ly.rename(columns={"tests": "actual_2025"})
ty = cnt[cnt.month.str.startswith("2026")].rename(columns={"tests": "actual_2026"})

# 4. Join everything on city + month (LEFT from plan, so a missing month shows up as NaN, not vanishes)
t = plan.merge(ly, on=["city", "month"], how="left").merge(ty, on=["city", "month"], how="left")
assert len(t) == len(plan), "join changed the row count"

# 5. The formulas
t["plan_over_2025"]   = t.plan_2026 / t.actual_2025            # CHECK: should be about 1.18
t["growth_pct"]       = (t.actual_2026 / t.actual_2025 - 1) * 100
t["vs_plan_pct"]      = (t.actual_2026 / t.plan_2026 - 1) * 100

pd.set_option("display.width", 200)
print("CHECK 1: is plan = 2025 count x 1.18 ?   (every cell should be near 1.18)")
print(t.pivot(index="month", columns="city", values="plan_over_2025").round(3))
print("\nCHECK 2: actual 2026 vs 2025, growth %")
print(t.pivot(index="month", columns="city", values="growth_pct").round(1))

# 6. Half-year totals and the gap in points, by city (only if both halves are present)
h = t.groupby("city")[["actual_2025", "actual_2026", "plan_2026"]].sum(min_count=1)
h["growth_pct"] = (h.actual_2026 / h.actual_2025 - 1) * 100
h["gap_points"] = (h.actual_2026 - h.plan_2026) / h.actual_2025.sum() * 100
print("\nH1 by city"); print(h.round(1))
tot = h[["actual_2025", "actual_2026", "plan_2026"]].sum()
print(f"\nNETWORK  H1 2025 = {tot.actual_2025:,.0f}   H1 2026 = {tot.actual_2026:,.0f}   plan = {tot.plan_2026:,.0f}")
print(f"growth = {(tot.actual_2026/tot.actual_2025-1)*100:.1f}%   plan growth = {(tot.plan_2026/tot.actual_2025-1)*100:.1f}%   gap = {h.gap_points.sum():.1f} points")

CHECK 1: is plan = 2025 count x 1.18 ?   (every cell should be near 1.18)
city     Ahmedabad  Bengaluru  Chennai  Hyderabad  Jaipur   Pune
month                                                           
2026-01      1.181      1.203    1.174      1.181   1.161  1.193
2026-02      1.182      1.203    1.173      1.181   1.162  1.193
2026-03      1.182      1.202    1.172      1.181   1.160  1.192
2026-04      1.182      1.204    1.172      1.182   1.163  1.192
2026-05      1.183      1.204    1.172      1.182   1.163  1.193
2026-06      1.181      1.202    1.173      1.183   1.162  1.191

CHECK 2: actual 2026 vs 2025, growth %
city     Ahmedabad  Bengaluru  Chennai  Hyderabad  Jaipur  Pune
month                                                          
2026-01        9.1        7.1      3.7        4.9   -11.6  10.9
2026-02       15.9       18.4     14.2        8.5   -15.7   4.7
2026-03       14.3       -0.4     -5.6       10.0   -16.7  16.2
2026-04        5.4       10.6     -2.9        

In [12]:
plan.head()

,city,month,plan_2026
0,Ahmedabad,2026-01,15119
1,Bengaluru,2026-01,20380
2,Chennai,2026-01,15474
3,Hyderabad,2026-01,14840
4,Jaipur,2026-01,8853


#Step 4: which counting rule reproduces the 5% and the plan's baseline?
Run from repo root:  python src/step4_count_rule.py


In [13]:
import pathlib

CLEAN = pathlib.Path("workbench_path")


In [14]:

import pandas as pd

PLAN_TOTAL = plan.drop(columns=["city", "month"]).sum().sum()


if "status_at_export" not in df.columns:
    df["status_at_export"] = pd.NA
df["booking_id"] = df["booking_id"].astype(str).str.strip()

# 2. Collections: stack all files, remove double scans (they are exact duplicate rows)
files = sorted((workbench_path / "collections").glob("*.csv.gz"))
print(f"found {len(files)} collections files")                         # expect 108
coll = pd.concat([pd.read_csv(p, dtype=str) for p in files], ignore_index=True)
n_in = len(coll)
coll = coll.drop_duplicates()
print(f"collections rows {n_in:,} -> {len(coll):,} after removing {n_in-len(coll):,} double scans")
dups = coll.booking_id.duplicated().sum()
print("PASS" if dups == 0 else "FAIL", "one collection row per booking after dedupe:", dups, "left")
coll = coll.drop_duplicates("booking_id")                                # safety: keep first if any remain

# 3. LEFT join, bookings -> collections. Row count must not change.
n_before = len(df)
df = df.merge(coll[["booking_id", "status"]].rename(columns={"status": "coll_status"}),
            on="booking_id", how="left", validate="many_to_one")
print("PASS" if len(df) == n_before else "FAIL", "join did not change row count:", n_before, "->", len(df))
df["coll_status"] = df.coll_status.fillna("no_collection_row")
print(df.drop_duplicates("booking_id").coll_status.value_counts().to_string())   # booking-level view

# 4. Drop old nova copies of rescheduled visits (decided earlier)
df = df[~df.status_at_export.eq("rescheduled").fillna(False).astype(bool)]

# 5. The candidate rules, each as a filter on test rows
rules = {
    "A  all booking-test rows":            df,
    "B  collected only":                   df[df.coll_status == "collected"],
    "C  collected + no_show":              df[df.coll_status.isin(["collected", "no_show"])],
}
h25 = ("2025-01-01", "2025-06-30 23:59:59"); h26 = ("2026-01-01", "2026-06-30 23:59:59")
rows = []
for name, d in rules.items():
    a = d.slot_at.between(*h25).sum(); c = d.slot_at.between(*h26).sum()
    rows.append({"rule": name, "H1_2025": a, "H1_2026": c, "growth_pct": (c / a - 1) * 100,
                 "plan_total / H1_2025": PLAN_TOTAL / a})
res = pd.DataFrame(rows).set_index("rule")
pd.set_option("display.width", 200)
print(f"\nPlan H1 2026 total = {PLAN_TOTAL:,.0f}   (plan growth is 18% if plan_total / H1_2025 = 1.18)")
print(res.round(3))
res.to_csv(workbench_path / "count_rule_comparison.csv")

found 108 collections files
collections rows 552,492 -> 549,598 after removing 2,894 double scans
PASS one collection row per booking after dedupe: 0 left
PASS join did not change row count: 1309199 -> 1309199
coll_status
collected            468320
no_show               78819
no_collection_row     18318

Plan H1 2026 total = 515,740   (plan growth is 18% if plan_total / H1_2025 = 1.18)
                          H1_2025  H1_2026  growth_pct  plan_total / H1_2025
rule                                                                        
A  all booking-test rows   435430   457230       5.007                 1.184
B  collected only          360725   378529       4.936                 1.430
C  collected + no_show     422451   440148       4.189                 1.221


In [15]:
df

,booking_id,created_at,slot_at,city,clinic_id,channel,source,patient_id,patient_name,test_code,test_price,home_fee,source_file,system,month,pairs,status_at_export,coll_status
0,BK-AHM-202501-000001,2025-01-08 03:16:00,2025-01-09 08:26:00,Ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T009,800,0,bookings_2025-01.csv.gz,legacy,2025-01,2025-01_Ahmedabad,<NA>,collected
1,BK-AHM-202501-000001,2025-01-08 03:16:00,2025-01-09 08:26:00,Ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T004,650,0,bookings_2025-01.csv.gz,legacy,2025-01,2025-01_Ahmedabad,<NA>,collected
2,BK-AHM-202501-000001,2025-01-08 03:16:00,2025-01-09 08:26:00,Ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T007,550,0,bookings_2025-01.csv.gz,legacy,2025-01,2025-01_Ahmedabad,<NA>,collected
3,BK-AHM-202501-000002,2025-01-03 05:18:00,2025-01-06 13:22:00,Ahmedabad,AHM-04,clinic,app,PAHM0031384,Nisha Iyer,T004,650,0,bookings_2025-01.csv.gz,legacy,2025-01,2025-01_Ahmedabad,<NA>,collected
4,BK-AHM-202501-000003,2025-01-27 10:46:00,2025-01-28 13:37:00,Ahmedabad,AHM-01,home,app,PAHM0027001,Sneha Menon,T003,450,150,bookings_2025-01.csv.gz,legacy,2025-01,2025-01_Ahmedabad,<NA>,collected
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1309194,nova_jai_0055752,2026-06-07 02:29:00,2026-06-12 11:18:00,Jaipur,JAI-02,home,phone,PJAI0017296,NaN,T007,550,150,bookings_2026-06.jsonl.gz,nova,2026-06,2026-06_Jaipur,<NA>,no_collection_row
1309195,nova_jai_0055752,2026-06-07 02:29:00,2026-06-12 11:18:00,Jaipur,JAI-02,home,phone,PJAI0017296,NaN,T006,650,150,bookings_2026-06.jsonl.gz,nova,2026-06,2026-06_Jaipur,<NA>,no_collection_row
1309196,nova_jai_0055753,2026-06-05 07:51:00,2026-06-06 10:25:00,Jaipur,JAI-03,clinic,phone,PJAI0027566,NaN,T006,650,0,bookings_2026-06.jsonl.gz,nova,2026-06,2026-06_Jaipur,<NA>,no_collection_row
1309197,nova_jai_0055753,2026-06-05 07:51:00,2026-06-06 10:25:00,Jaipur,JAI-03,clinic,phone,PJAI0027566,NaN,T002,120,0,bookings_2026-06.jsonl.gz,nova,2026-06,2026-06_Jaipur,<NA>,no_collection_row


In [16]:
"""Step 5: clean the bookings, one rule at a time, logging the effect of each rule on the headline.
Run from repo root:  python src/step5_clean.py
Set COUNT_RULE after you have seen Step 4:  "A" all rows | "B" collected only | "C" collected + no_show"""
import pathlib, numpy as np, pandas as pd

COUNT_RULE = "A"                       # <-- change after Step 4
H25 = ("2025-01-01", "2025-06-30 23:59:59"); H26 = ("2026-01-01", "2026-06-30 23:59:59")

def headline(d):
    a = int(d.slot_at.between(*H25).sum()); c = int(d.slot_at.between(*H26).sum())
    return a, c, ((c / a - 1) * 100 if a else np.nan)

n0 = len(df)


df["coll_status"] = df.coll_status.fillna("no_collection_row")

# ---- flags (keep the row, mark it) -----------------------------------------------
df["flag_blank_test_code"] = df.test_code.isna() & df.test_price.notna()       # legacy-style blank code, price valid
print("flag only: blank test_code with valid price:", int(df.flag_blank_test_code.sum()), "rows (kept; the plan counted them)")
print("report only: fully duplicated rows:", int(df.drop(columns="source_file").duplicated().sum()), "(not removed; look before deciding)")

# ---- rules that remove rows ------------------------------------------------------
log, rejects = [], []
def apply_rule(df, name, reason, reject_mask):
    a0, c0, g0 = headline(df)
    out, rej = df[~reject_mask], df[reject_mask].assign(reject_rule=name)
    a1, c1, g1 = headline(out)
    assert len(df) == len(out) + len(rej)                                   # input = clean + rejected
    log.append({"rule": name, "reason": reason, "rows_in": len(df), "rows_rejected": len(rej), "rows_out": len(out),
                "H1_2025_after": a1, "H1_2026_after": c1, "growth_before": round(g0, 2), "growth_after": round(g1, 2), "moved_pts": round(g1 - g0, 2)})
    rejects.append(rej)
    return out

b = apply_rule(df, "R1 rescheduled originals (nova)", "nova keeps the old copy of a moved visit; legacy overwrote it, so one visit = one booking",
               df.status_at_export.eq("rescheduled").fillna(False).astype(bool))
b = apply_rule(b, "R2 nova bookings with no tests", "placeholder row for a booking whose test list is empty; no test was ordered",
               (b.system == "nova") & b.test_code.isna() & b.test_price.isna())
if COUNT_RULE == "B":
    b = apply_rule(b, "R3 count rule B", "count performed tests only: collected", b.coll_status != "collected")
elif COUNT_RULE == "C":
    b = apply_rule(b, "R3 count rule C", "count collected plus no-show; drop cancelled / no collection row", ~b.coll_status.isin(["collected", "no_show"]))

# ---- results ---------------------------------------------------------------------
L = pd.DataFrame(log); pd.set_option("display.width", 220); pd.set_option("display.max_columns", 20)
print(f"\nCOUNT_RULE = {COUNT_RULE}\n", L.drop(columns="reason").to_string(index=False))
a, c, g = headline(b)
print(f"\nHEADLINE after cleaning: H1 2025 = {a:,}  H1 2026 = {c:,}  growth = {g:.2f}%")
rej = pd.concat(rejects) if rejects else pd.DataFrame()
print(f"RECONCILE: input {n0:,} = clean {len(b):,} + rejected {len(rej):,} ->", "PASS" if n0 == len(b) + len(rej) else "FAIL")

b.to_csv(workbench_path / "bookings_clean.csv.gz", index=False, compression="gzip")
rej.to_csv(workbench_path / "rejects_step5.csv.gz", index=False, compression="gzip")
L.to_csv(workbench_path / "decisions_log_auto.csv", index=False)
print("saved bookings_clean.csv.gz, rejects_step5.csv.gz, decisions_log_auto.csv")

flag only: blank test_code with valid price: 1289 rows (kept; the plan counted them)
report only: fully duplicated rows: 0 (not removed; look before deciding)

COUNT_RULE = A
                            rule  rows_in  rows_rejected  rows_out  H1_2025_after  H1_2026_after  growth_before  growth_after  moved_pts
R1 rescheduled originals (nova)  1309199              0   1309199         435430         457230           5.01          5.01        0.0
 R2 nova bookings with no tests  1309199              0   1309199         435430         457230           5.01          5.01        0.0

HEADLINE after cleaning: H1 2025 = 435,430  H1 2026 = 457,230  growth = 5.01%
RECONCILE: input 1,309,199 = clean 1,309,199 + rejected 0 -> PASS
saved bookings_clean.csv.gz, rejects_step5.csv.gz, decisions_log_auto.csv


In [17]:
"""Step 5b: turn patient ids into PEOPLE. Adds person_id to bookings_clean.csv.gz.
Rule: two patient rows are the same person if they share a valid phone number and the same name (case and spacing ignored).
Rows with a missing or placeholder phone are never merged. Run AFTER step5_clean.py, BEFORE step6_tree.py.
Run from repo root:  python src/step5b_patients.py"""
import sys, re, pathlib, numpy as np, pandas as pd

#RAW, CLEAN = pathlib.Path("data/raw"), pathlib.Path("data/clean")
pf = workbench_path / "patients.csv"
if not pf.exists(): sys.exit(f"STOP: {pf} not found. A cold run must not skip the people step.")
p = pd.read_csv(pf, dtype=str); p["patient_id"] = p.patient_id.str.strip()
need = {"patient_id", "patient_name", "phone"}
if not need <= set(p.columns): sys.exit(f"STOP: patient master needs columns {need}; found {list(p.columns)}")
print(f"patient master: {len(p):,} rows, {p.patient_id.nunique():,} distinct patient ids")

# ---- normalise ----
digits = p.phone.fillna("").str.replace(r"\D", "", regex=True).str[-10:]
p["phone_norm"] = digits
p["phone_ok"] = (digits.str.len() == 10) & ~digits.str.fullmatch(r"(\d)\1{9}") & ~digits.str.startswith("000")
p["name_norm"] = p.patient_name.fillna("").str.upper().str.replace(r"\s+", " ", regex=True).str.strip()
use = p.phone_ok & (p.name_norm != "")
p["key"] = np.where(use, p.phone_norm + "|" + p.name_norm, "SINGLE|" + p.patient_id)
print(f"valid phone and name on {use.mean():.1%} of rows; the rest stay as separate people (never merged)")

# ---- the decision, with its evidence ----
p["person_id"] = p.groupby("key").patient_id.transform("min")
n_ids, n_people = p.patient_id.nunique(), p.person_id.nunique()
print(f"RULE phone + name: {n_ids:,} patient ids -> {n_people:,} people ({n_ids-n_people:,} ids merged, {(n_ids-n_people)/n_ids:.1%})")
if "dob" in p.columns:
    d = p.dob.where(~p.dob.fillna("").str.startswith("1900"))                                # 1900-01-01 is a placeholder
    g = p[use].assign(dob=d).groupby("key").dob.nunique(); conflict = int((g > 1).sum())
    alt = p.assign(k2=p.key + "|" + d.fillna("?")).k2.nunique()
    print(f"RISK CHECK: {conflict:,} merged groups contain more than one date of birth (possible different people). Adding dob to the rule would give {alt:,} people instead of {n_people:,}.")
same_name = p[use].groupby("name_norm").patient_id.nunique().sort_values(ascending=False)
print(f"for scale: the most common name ({same_name.index[0]}) sits on {same_name.iloc[0]} ids, which is why NAME ALONE is never used")

# ---- attach to bookings ----
#b = pd.read_csv(CLEAN / "bookings_clean.csv.gz", parse_dates=["slot_at"], dtype={"booking_id": "string", "patient_id": "string"}).drop(columns=["person_id"], errors="ignore")
n0 = len(b); b["patient_id"] = b.patient_id.str.strip()
b = b.merge(p.drop_duplicates("patient_id")[["patient_id", "person_id"]], on="patient_id", how="left", validate="many_to_one")
assert len(b) == n0, "join changed the row count"
miss = b.person_id.isna(); print(f"PASS join did not change row count ({n0:,})" ); print(("PASS" if miss.sum() == 0 else "WARN") + f" booking rows whose patient id is not in the master: {int(miss.sum()):,} (kept, using the patient id as the person)")
b["person_id"] = b.person_id.fillna(b.patient_id)
for h, (s, e) in {"H1_2025": ("2025-01-01", "2025-06-30 23:59:59"), "H1_2026": ("2026-01-01", "2026-06-30 23:59:59")}.items():
    d_ = b[b.slot_at.between(s, e)]; print(f"  {h}: people counted as ids {d_.patient_id.nunique():,} -> as persons {d_.person_id.nunique():,}")
b.to_csv(workbench_path / "bookings_clean.csv.gz", index=False, compression="gzip")
p.to_csv(workbench_path / "patients_clean.csv.gz", index=False, compression="gzip"); print("saved bookings_clean.csv.gz (now with person_id) and patients_clean.csv.gz")
print("LOG IT: rule, ids merged, effect on 'people' in each half (above), and the dob risk check.")

patient master: 309,217 rows, 309,217 distinct patient ids
valid phone and name on 100.0% of rows; the rest stay as separate people (never merged)
RULE phone + name: 309,217 patient ids -> 303,153 people (6,064 ids merged, 2.0%)
RISK CHECK: 0 merged groups contain more than one date of birth (possible different people). Adding dob to the rule would give 303,153 people instead of 303,153.
for scale: the most common name (NISHA GUPTA) sits on 586 ids, which is why NAME ALONE is never used
PASS join did not change row count (1,309,199)
PASS booking rows whose patient id is not in the master: 0 (kept, using the patient id as the person)
  H1_2025: people counted as ids 94,477 -> as persons 94,477
  H1_2026: people counted as ids 98,025 -> as persons 98,025
saved bookings_clean.csv.gz (now with person_id) and patients_clean.csv.gz
LOG IT: rule, ids merged, effect on 'people' in each half (above), and the dob risk check.


In [24]:
"""Step 6: the tree. Tests = people x visits per person x tests per visit, for H1 2025 and H1 2026,
then split by city and by channel. Run from repo root:  python src/step6_tree.py"""
import pathlib, numpy as np, pandas as pd

b = pd.read_csv(workbench_path / "bookings_clean.csv.gz", parse_dates=["slot_at"], dtype={"booking_id": "string", "patient_id": "string", "person_id": "string"})
PERSON = "person_id" if "person_id" in b.columns else "patient_id"
if PERSON == "patient_id":
    print("WARNING: no person_id yet (Step 5b not run). 'People' = distinct patient ids, so duplicates inflate it.\n")

b["half"] = np.where(b.slot_at.between("2025-01-01", "2025-06-30 23:59:59"), "H1_2025",
            np.where(b.slot_at.between("2026-01-01", "2026-06-30 23:59:59"), "H1_2026", "other"))
b = b[b.half != "other"].copy(); b["all"] = "Network"
print(f"{len(b):,} test rows in the two halves (cleaned, count rule already applied in Step 5)")

def tree(d, by):
    g = d.groupby(by + ["half"]).agg(tests=("booking_id", "size"), visits=("booking_id", "nunique"), people=(PERSON, "nunique"))
    g["visits_per_person"] = g.visits / g.people
    g["tests_per_visit"] = g.tests / g.visits
    return g

def growth(g):
    a, c = g.xs("H1_2025", level="half"), g.xs("H1_2026", level="half")
    out = (c / a - 1) * 100
    for br in ["people", "visits_per_person", "tests_per_visit"]:                      # log split: the three add up to the total
        out[br + "_pts"] = np.log(c[br] / a[br]) / np.log(c["tests"] / a["tests"]) * out["tests"]
    return out

# ---- 1. network tree -------------------------------------------------------------
net = tree(b, ["all"]); gn = growth(net)
print("\nTHE TREE, network"); print(net.round(3).to_string())
r = gn.iloc[0]
print(f"\ngrowth %:  tests {r.tests:+.2f} | people {r.people:+.2f} | visits/person {r.visits_per_person:+.2f} | tests/visit {r.tests_per_visit:+.2f}")
mult = (1 + r.people/100) * (1 + r.visits_per_person/100) * (1 + r.tests_per_visit/100) - 1
print("CHECK branches multiply back to the total:", "PASS" if abs(mult * 100 - r.tests) < 1e-6 else "FAIL", f"({mult*100:.2f} vs {r.tests:.2f})")
print(f"points of the {r.tests:+.2f}% each branch carries: people {r.people_pts:+.2f}, visits/person {r.visits_per_person_pts:+.2f}, tests/visit {r.tests_per_visit_pts:+.2f}")
print("sum of points equals total:", "PASS" if abs(r.people_pts + r.visits_per_person_pts + r.tests_per_visit_pts - r.tests) < 1e-6 else "FAIL")

# ---- 2. tree by city, and the gap against the plan -------------------------------
cityg = growth(tree(b, ["city"]))
plan = pd.read_csv(workbench_path / "plan_2026_H1.csv")
plan["city"] = plan.city.replace({"Bangalore": "Bengaluru"})
plan_city = plan.drop(columns="metric").set_index("city").sum(axis=1).rename("plan_2026")
t = tree(b, ["city"]); a25, a26 = t.xs("H1_2025", level="half").tests, t.xs("H1_2026", level="half").tests
cityg["tests_2025"], cityg["tests_2026"] = a25, a26
cityg = cityg.join(plan_city)
cityg["gap_points"] = (cityg.tests_2026 - cityg.plan_2026) / a25.sum() * 100        # shares of the network gap; sums to the total
print("\nBY CITY: growth % of each branch, and gap points against plan")
print(cityg[["tests_2025", "tests_2026", "plan_2026", "tests", "people", "visits_per_person", "tests_per_visit", "gap_points"]].round(1).sort_values("gap_points").to_string())
print(f"\nNETWORK gap vs plan = {cityg.gap_points.sum():.1f} points  (actual {r.tests:+.1f}% vs plan {(cityg.plan_2026.sum()/a25.sum()-1)*100:+.1f}%)")

# ---- 3. city x channel: where the lost growth sits --------------------------------
cc = b.groupby(["city", "channel", "half"]).size().unstack("half").fillna(0)
cc["growth_pct"] = (cc.H1_2026 / cc.H1_2025 - 1) * 100
cc["pts_of_network_growth"] = (cc.H1_2026 - cc.H1_2025) / b[b.half == "H1_2025"].shape[0] * 100
print("\nBY CITY x CHANNEL (the plan has no channel split, so this is growth, not gap)")
print(cc.round(1).sort_values("pts_of_network_growth").to_string())
short = gn[["people", "visits_per_person", "tests_per_visit"]].iloc[0].idxmin()
print(f"\nWEAKEST BRANCH (lowest growth): {short}")

net.to_csv(workbench_path / "tree_network.csv"); cityg.to_csv(workbench_path / "tree_by_city.csv"); cc.to_csv(workbench_path / "city_channel_growth.csv")
print("saved tree_network.csv, tree_by_city.csv, city_channel_growth.csv")

/tmp/ipykernel_13292/603967027.py:5: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  b = pd.read_csv(workbench_path / "bookings_clean.csv.gz", parse_dates=["slot_at"], dtype={"booking_id": "string", "patient_id": "string", "person_id": "string"})


892,660 test rows in the two halves (cleaned, count rule already applied in Step 5)

THE TREE, network
                  tests  visits  people  visits_per_person  tests_per_visit
all     half                                                               
Network H1_2025  435430  189500   94477              2.006            2.298
        H1_2026  457230  194713   98025              1.986            2.348

growth %:  tests +5.01 | people +3.76 | visits/person -0.97 | tests/visit +2.20
CHECK branches multiply back to the total: PASS (5.01 vs 5.01)
points of the +5.01% each branch carries: people +3.78, visits/person -1.00, tests/visit +2.23
sum of points equals total: PASS

BY CITY: growth % of each branch, and gap points against plan
           tests_2025  tests_2026  plan_2026  tests  people  visits_per_person  tests_per_visit  gap_points
city                                                                                                       
Chennai         80094       80676      939

In [27]:
"""Step 6b: where did the growth go? Splits the result by channel (clinic/home), booking source (app/phone/walkin),
city x channel and city x source, and looks at it by month, quarter and half-year.
Needs Step 5 and Step 6 run.  Run from repo root:  python src/step6b_segments.py"""
import pathlib, numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt

RAW, CLEAN, OUT = pathlib.Path(workbench_path), pathlib.Path(workbench_path), pathlib.Path("slides"); OUT.mkdir(exist_ok=True)
PERSON = "person_id" if "person_id" in b.columns else "patient_id"
b["half"] = np.select([b.slot_at.between("2025-01-01", "2025-06-30 23:59:59"), b.slot_at.between("2025-07-01", "2025-12-31 23:59:59"),
                       b.slot_at.between("2026-01-01", "2026-06-30 23:59:59")], ["H1_2025", "H2_2025", "H1_2026"], "other")
b["q"] = b.slot_at.dt.to_period("Q").astype(str); b["m"] = b.slot_at.dt.strftime("%Y-%m"); b["Network"] = "Network"
H = b[b.half.isin(["H1_2025", "H1_2026"])]
T25, T26 = (H.half == "H1_2025").sum(), (H.half == "H1_2026").sum()
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 20)
print(f"Network: H1 2025 {T25:,} -> H1 2026 {T26:,} = {(T26/T25-1)*100:+.2f}%")
cs = pd.crosstab(H.channel, H.source, normalize="all").mul(100).round(1); print("\nMix of tests, % of all (channel x source), H1 2025 + H1 2026\n", cs.to_string())

# ---- 1. segment table: growth, share, points of the network growth, and the tree inside each segment --------------------
def seg_table(dim):
    g = H.groupby([dim, "half"]).agg(tests=("booking_id", "size"), visits=("booking_id", "nunique"), people=(PERSON, "nunique")).unstack("half").fillna(0)
    t = pd.DataFrame(index=g.index); a, c = (lambda k: g[(k, "H1_2025")]), (lambda k: g[(k, "H1_2026")])
    t["tests_2025"], t["tests_2026"] = a("tests"), c("tests"); t["growth_%"] = (t.tests_2026 / t.tests_2025 - 1) * 100
    t["share_2025_%"], t["share_2026_%"] = t.tests_2025 / T25 * 100, t.tests_2026 / T26 * 100
    t["pts_of_network_growth"] = (t.tests_2026 - t.tests_2025) / T25 * 100                     # these add up to the network growth
    t["people_%"] = (c("people") / a("people") - 1) * 100
    t["visits_per_person_%"] = ((c("visits") / c("people")) / (a("visits") / a("people")) - 1) * 100
    t["tests_per_visit_%"] = ((c("tests") / c("visits")) / (a("tests") / a("visits")) - 1) * 100
    return t.round(2)
tabs = {}
for dim in ["channel", "source"]:
    tabs[dim] = seg_table(dim); print(f"\nBY {dim.upper()}  (pts_of_network_growth adds up to {(T26/T25-1)*100:+.2f})\n", tabs[dim].to_string())
    print("  CHECK points add up to network growth:", "PASS" if abs(tabs[dim].pts_of_network_growth.sum() - (T26 / T25 - 1) * 100) < 0.05 else "FAIL")
    tabs[dim].to_csv(CLEAN / f"segments_{dim}.csv")

# ---- 2. gap against the plan, by segment. ASSUMPTION: each city's plan is spread over its segments in last year's mix ------
plan = pd.read_csv(workbench_path /"plan_2026_H1.csv"); plan["city"] = plan.city.replace({"Bangalore": "Bengaluru"})
plan_city = plan.drop(columns="metric").set_index("city").sum(axis=1)
def gap_alloc(dims):
    x = H.groupby(["city"] + [d for d in dims if d != "city"] + ["half"]).size().unstack("half").fillna(0)
    x["plan_seg"] = x.index.get_level_values("city").map(plan_city) * x.H1_2025 / x.groupby(level="city").H1_2025.transform("sum")
    x["gap_pts"] = (x.H1_2026 - x.plan_seg) / T25 * 100
    o = x.groupby(level=dims).agg(tests_2025=("H1_2025", "sum"), tests_2026=("H1_2026", "sum"), plan_allocated=("plan_seg", "sum"), gap_pts=("gap_pts", "sum"))
    o["growth_%"] = (o.tests_2026 / o.tests_2025 - 1) * 100; o["plan_growth_%"] = (o.plan_allocated / o.tests_2025 - 1) * 100
    return o.round(2)
G = {k: gap_alloc(v) for k, v in {"channel": ["channel"], "source": ["source"], "city_channel": ["city", "channel"], "city_source": ["city", "source"]}.items()}
print("\nGAP AGAINST PLAN, with each city's plan spread over its segments in last year's mix (an assumption: the plan has no channel or source split)")
for k, v in G.items(): print(f"\n{k}  (gap points, add up to {v.gap_pts.sum():.1f})\n", v.sort_values("gap_pts").head(8 if "_" in k else 5).to_string()); v.to_csv(CLEAN / f"gap_{k}.csv")

# ---- 3. time: month, quarter, half --------------------------------------------------------------------------------------
def yoy(dim, col, a, c):
    t = b.groupby([dim, col]).size().unstack(col); return ((t.get(c) / t.get(a) - 1) * 100) if (c in t and a in t) else pd.Series(np.nan, index=t.index)
QT = pd.concat([pd.DataFrame({"Q1_yoy_%": yoy(d, "q", "2025Q1", "2026Q1"), "Q2_yoy_%": yoy(d, "q", "2025Q2", "2026Q2")}) for d in ["Network", "channel", "source"]]).round(1)
print("\nYEAR-ON-YEAR BY QUARTER (did the shortfall start in Q2?)\n", QT.to_string())
mo = pd.DataFrame({d: ((b[b.channel == d] if d != "Network" else b).groupby("m").size()) for d in ["Network", "clinic", "home"]})
my = pd.DataFrame({d: [(mo.loc[f"2026-0{i}", d] / mo.loc[f"2025-0{i}", d] - 1) * 100 if (f"2026-0{i}" in mo.index and f"2025-0{i}" in mo.index) else np.nan for i in range(1, 7)] for d in mo}, index=[f"2026-0{i}" for i in range(1, 7)]).round(1)
print("\nMONTHLY YEAR-ON-YEAR %, network and channel\n", my.to_string())
hh = b[b.half != "other"].groupby(["channel", "half"]).size().unstack("half").reindex(columns=["H1_2025", "H2_2025", "H1_2026"])
hh.loc["ALL"] = hh.sum(); hh["H2_vs_H1_2025_%"] = (hh.H2_2025 / hh.H1_2025 - 1) * 100; hh["H1_2026_vs_H2_2025_%"] = (hh.H1_2026 / hh.H2_2025 - 1) * 100
print("\nHALF-YEAR SEQUENCE (seasonality check: the plan assumed a flat +18% on every month)\n", hh.round(1).to_string())

# ---- 4. tests per visit and packages ------------------------------------------------------------------------------------
b["pk"] = b.test_code.astype("string").str.startswith("PK").fillna(False)
pk = b[b.half.isin(["H1_2025", "H1_2026"])].groupby(["channel", "half"]).agg(package_share_of_rows_pct=("pk", lambda s: s.mean() * 100), tests_per_visit=("booking_id", lambda s: len(s) / s.nunique())).round(2)
print("\nPACKAGES AND TESTS PER VISIT (a package is one row; more packages can lower tests per visit without fewer patients)\n", pk.to_string())

# ---- 5. the headline lines ------------------------------------------------------------------------------------------------
worst = pd.concat([G["channel"].assign(kind="channel"), G["source"].assign(kind="source")]).gap_pts.idxmin()
wcc = G["city_channel"].gap_pts.idxmin()
print(f"\nWHERE IT SITS: biggest gap among channel/source: {worst} ({pd.concat([G['channel'], G['source']]).gap_pts.min():.1f} pts); biggest city x channel cell: {wcc} ({G['city_channel'].gap_pts.min():.1f} pts)")
print("State the plan-allocation assumption on the slide if you quote any of these gap numbers.")

# ---- 6. chart ------------------------------------------------------------------------------------------------------------
lab = pd.concat([G["channel"].gap_pts.rename(lambda x: f"channel: {x}"), G["source"].gap_pts.rename(lambda x: f"source: {x}")]).sort_values()
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].barh(lab.index, lab.values, color="#D63A6A"); ax[0].set_xlabel(f"Gap vs plan, points of H1 2025 tests ({T25:,}); plan spread by last year's mix"); ax[0].set_title("Where the gap sits")
for c_ in ["Network", "clinic", "home"]: ax[1].plot(my.index, my[c_], marker="o", lw=2.5 if c_ == "Network" else 1.5, label=c_)
ax[1].axhline(0, color="#1A1440", lw=1); ax[1].set_title("Monthly growth vs same month last year, %"); ax[1].legend(frameon=False); ax[1].tick_params(axis="x", labelsize=8, rotation=30)
plt.tight_layout(); fig.savefig(OUT / "segments.png", dpi=180); plt.close(); print("saved slides/segments.png and data/clean/segments_*.csv, gap_*.csv")

Network: H1 2025 435,430 -> H1 2026 457,230 = +5.01%

Mix of tests, % of all (channel x source), H1 2025 + H1 2026
 source    app  phone  walkin
channel                     
clinic   32.8   18.4     9.2
home     21.5   12.1     6.0

BY CHANNEL  (pts_of_network_growth adds up to +5.01)
          tests_2025  tests_2026  growth_%  share_2025_%  share_2026_%  pts_of_network_growth  people_%  visits_per_person_%  tests_per_visit_%
channel                                                                                                                                       
clinic       286692      252678    -11.86         65.84         55.26                  -7.81     -8.31                -6.08               2.35
home         148738      204552     37.53         34.16         44.74                  12.82     26.35                 6.82               1.90
  CHECK points add up to network growth: PASS

BY SOURCE  (pts_of_network_growth adds up to +5.01)
         tests_2025  tests_2026  growth_% 

In [28]:
"""Step 6c: fill every box of the handbook's two trees, for H1 2025 and H1 2026.
  Tree 1 (industry page):  Revenue = Patients x Tests per patient x Revenue per test
  Tree 2 (Brief A page):   Test volume = Patients served x Visits per patient x Tests per visit
                           Patients served = New patients + Returning patients      Revenue = Test volume x Revenue per test
Needs Step 5 (bookings_clean). For New vs Returning it uses signup_date from the patient master if data/raw/patients/patients.csv.gz exists.
Run from repo root:  python src/step6c_tree_boxes.py"""
# import pathlib, numpy as np, pandas as pd

# RAW, CLEAN = pathlib.Path("data/raw"), pathlib.Path("data/clean")
W = {"H1_2025": ("2025-01-01", "2025-06-30 23:59:59"), "H1_2026": ("2026-01-01", "2026-06-30 23:59:59")}
b = pd.read_csv(CLEAN / "bookings_clean.csv.gz", parse_dates=["slot_at"], dtype={"booking_id": "string", "patient_id": "string", "person_id": "string"})
PERSON = "person_id" if "person_id" in b.columns else "patient_id"
b["price"] = pd.to_numeric(b.test_price, errors="coerce")
if PERSON == "patient_id": print("WARNING: no person_id yet, so 'patients' = distinct patient ids (duplicates inflate it).\n")

# ---- new vs returning ----------------------------------------------------------------
signup = None; pm = RAW / "patients" / "patients.csv.gz"
if pm.exists():
    p = pd.read_csv(pm, dtype=str, usecols=lambda c: c in ("patient_id", "signup_date"))
    d = pd.to_datetime(p.signup_date, errors="coerce")
    if d.isna().mean() > 0.05: d = pd.to_datetime(p.signup_date, errors="coerce", dayfirst=True)
    p["signup"] = d; print(f"patient master: {len(p):,} rows, signup_date unreadable in {d.isna().mean():.1%}")
    if PERSON == "person_id":
        mp = b[["patient_id", "person_id"]].drop_duplicates("patient_id").merge(p[["patient_id", "signup"]], on="patient_id", how="left")
        signup = mp.groupby("person_id").signup.min()
    else: signup = p.drop_duplicates("patient_id").set_index("patient_id").signup
else: print("NOTE: patient master not found, so new/returning uses the fallback below (H1 2025 cannot be split: the data starts in Jan 2025).\n")

rows = {}
for h, (s, e) in W.items():
    d = b[b.slot_at.between(s, e)]; ppl = d[PERSON].dropna().unique(); tests, visits, n = len(d), d.booking_id.nunique(), len(ppl)
    r = {"patients_served": n, "visits": visits, "tests": tests, "visits_per_patient": visits / n, "tests_per_visit": tests / visits,
         "tests_per_patient": tests / n, "revenue_booked_Rs_crore": d.price.sum() / 1e7, "revenue_per_test_Rs": d.price.sum() / tests}
    if signup is not None:
        sg = pd.Series(ppl).map(signup); r["new_patients"] = int((sg >= pd.Timestamp(s)).sum()); r["returning_patients"] = int((sg < pd.Timestamp(s)).sum()); r["signup_unknown"] = int(sg.isna().sum())
    elif h == "H1_2026":
        prev = set(b[b.slot_at.between(*W["H1_2025"])][PERSON].dropna()); r["returning_patients"] = int(sum(x in prev for x in ppl)); r["new_patients"] = n - r["returning_patients"]; r["signup_unknown"] = 0
    else: r["new_patients"] = r["returning_patients"] = r["signup_unknown"] = np.nan
    rows[h] = r
T = pd.DataFrame(rows); T["change_%"] = (T.H1_2026 / T.H1_2025 - 1) * 100
order = ["new_patients", "returning_patients", "patients_served", "visits_per_patient", "tests_per_visit", "tests", "tests_per_patient", "revenue_per_test_Rs", "revenue_booked_Rs_crore"]
pd.set_option("display.width", 200); pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
print("EVERY BOX, BOTH YEARS\n", T.loc[order].to_string())
a, c = T.H1_2025, T.H1_2026
ok = lambda x, y: "PASS" if abs(x - y) <= 1e-6 * max(1, abs(y)) else "FAIL"
print("\nCHECKS")
for h, col in (("H1_2025", a), ("H1_2026", c)):
    print(f"  {h}: tests = patients x visits/patient x tests/visit:", ok(col.patients_served * col.visits_per_patient * col.tests_per_visit, col.tests),
          "| revenue = tests x revenue/test:", ok(col.tests * col.revenue_per_test_Rs / 1e7, col.revenue_booked_Rs_crore),
          "| patients = new + returning:", ("n/a" if np.isnan(col.new_patients) else ok(col.new_patients + col.returning_patients + col.signup_unknown, col.patients_served)))
print("  note: revenue is BOOKED at list price (test_price). Billed revenue would come from invoices (Brief C data) and can differ.")
T.to_csv(CLEAN / "tree_boxes.csv"); print("saved data/clean/tree_boxes.csv")

/tmp/ipykernel_13292/684455834.py:11: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  b = pd.read_csv(CLEAN / "bookings_clean.csv.gz", parse_dates=["slot_at"], dtype={"booking_id": "string", "patient_id": "string", "person_id": "string"})


NOTE: patient master not found, so new/returning uses the fallback below (H1 2025 cannot be split: the data starts in Jan 2025).

EVERY BOX, BOTH YEARS
                             H1_2025     H1_2026  change_%
new_patients                    NaN  66,991.000       NaN
returning_patients              NaN  31,034.000       NaN
patients_served          94,477.000  98,025.000     3.755
visits_per_patient            2.006       1.986    -0.968
tests_per_visit               2.298       2.348     2.195
tests                   435,430.000 457,230.000     5.007
tests_per_patient             4.609       4.664     1.206
revenue_per_test_Rs         638.885     688.788     7.811
revenue_booked_Rs_crore      27.819      31.493    13.209

CHECKS
  H1_2025: tests = patients x visits/patient x tests/visit: PASS | revenue = tests x revenue/test: PASS | patients = new + returning: n/a
  H1_2026: tests = patients x visits/patient x tests/visit: PASS | revenue = tests x revenue/test: PASS | patients = new 

In [30]:
"""Optional but worth running once: do the RESULTS (tests the lab actually ran) agree with our booking-test counts?
This is the evidence for the approach-log row 'which table did we count from, and why not the other two'.
Run from repo root:  python src/step_results_crosscheck.py"""
# import sys, pathlib, numpy as np, pandas as pd
# RAW, CLEAN = pathlib.Path("data/raw"), pathlib.Path("data/clean")
files = sorted((RAW / "results").rglob("*.csv.gz")); print(f"{len(files)} results files")      # expect 108
res = pd.concat([pd.read_csv(f, dtype=str, usecols=["booking_id"]) for f in files], ignore_index=True)
res["booking_id"] = res.booking_id.str.strip(); print(f"{len(res):,} result rows, {res.booking_id.nunique():,} distinct bookings")

# b = pd.read_csv(CLEAN / "bookings_all.csv.gz", parse_dates=["slot_at"], dtype={"status_at_export": "string"})
if "status_at_export" not in b.columns: b["status_at_export"] = pd.NA
b["booking_id"] = b.booking_id.astype(str).str.strip()
b["orig"] = b.status_at_export.eq("rescheduled").fillna(False).astype(bool)
slot = b.drop_duplicates("booking_id").set_index("booking_id").slot_at
res["slot_at"] = res.booking_id.map(slot); orphan = res.slot_at.isna()
print(f"result rows whose booking id is not in bookings: {int(orphan.sum()):,} ({orphan.mean():.2%})")

H = {"H1_2025": ("2025-01-01", "2025-06-30 23:59:59"), "H1_2026": ("2026-01-01", "2026-06-30 23:59:59")}
bk = b[~b.orig]; out = {}
for k, (s, e) in H.items(): out[k] = {"bookings_rows (rule A)": int(bk.slot_at.between(s, e).sum()), "results_rows": int(res.slot_at.between(s, e).sum())}
T = pd.DataFrame(out).T; T["results_as_%_of_bookings"] = T.results_rows / T["bookings_rows (rule A)"] * 100
g = lambda c: (T.loc["H1_2026", c] / T.loc["H1_2025", c] - 1) * 100
pd.set_option("display.width", 200); print("\n", T.round(1).to_string())
print(f"\nGROWTH  bookings rows {g('bookings_rows (rule A)'):+.2f}%   results rows {g('results_rows'):+.2f}%")
c = pd.concat([bk.groupby("booking_id").size().rename("booking_rows"), res.groupby("booking_id").size().rename("result_rows")], axis=1).dropna()
print(f"on bookings that have results: booking rows == result rows for {(c.booking_rows == c.result_rows).mean():.1%} of {len(c):,} bookings")
print("READ IT: results cover tests actually run, so they exclude no-shows and cancellations. Growth that differs from the plan's basis is a reason NOT to count from results.")

108 results files
1,078,420 result rows, 473,905 distinct bookings
result rows whose booking id is not in bookings: 10,295 (0.95%)

          bookings_rows (rule A)  results_rows  results_as_%_of_bookings
H1_2025                  435430        355232                    81.600
H1_2026                  457230        372919                    81.600

GROWTH  bookings rows +5.01%   results rows +4.98%
on bookings that have results: booking rows == result rows for 97.0% of 466,406 bookings
READ IT: results cover tests actually run, so they exclude no-shows and cancellations. Growth that differs from the plan's basis is a reason NOT to count from results.


In [35]:
import pathlib
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment

# ---- EDIT THESE after you have seen the numbers ------------------------------------
BRIEF, GROUP = "A", "<N>"
CLAIM = None        # your final sentence, in quotes. Leave None to use the auto-draft.
CAVEAT = "<the one thing that would change the claim, and its size>"
# ------------------------------------------------------------------------------------

# Set up paths
CLEAN = pathlib.Path(workbench_path)
OUT = pathlib.Path("slides")
OUT.mkdir(exist_ok=True)

PURPLE, PINK, DARK, GREY = "#4E38AB", "#D63A6A", "#1A1440", "#9A96B5"

# Load data
b = pd.read_csv(CLEAN / "bookings_clean.csv.gz", parse_dates=["slot_at"])
b["half"] = np.where(
    b.slot_at.between("2025-01-01", "2025-06-30 23:59:59"), "H1_2025",
    np.where(b.slot_at.between("2026-01-01", "2026-06-30 23:59:59"), "H1_2026", "other")
)

plan = pd.read_csv(CLEAN / "plan_2026_H1.csv").melt(id_vars=["city", "metric"], var_name="month", value_name="plan")
plan["city"] = plan.city.replace({"Bangalore": "Bengaluru"})

# ---- Network & City Numbers -------------------------------------------------------
h = b[b.half != "other"].groupby(["city", "half"]).size().unstack("half").fillna(0)
h["plan"] = plan.groupby("city").plan.sum()
T25, T26, TPLAN = h.H1_2025.sum(), h.H1_2026.sum(), h.plan.sum()
h["growth"] = (h.H1_2026 / h.H1_2025 - 1) * 100
h["plan_growth"] = (h.plan / h.H1_2025 - 1) * 100
h["gap_pts"] = (h.H1_2026 - h.plan) / T25 * 100
g, pg, gap = (T26 / T25 - 1) * 100, (TPLAN / T25 - 1) * 100, h.gap_pts.sum()

print(f"NETWORK  H1 2025 {T25:,.0f} | H1 2026 {T26:,.0f} | plan {TPLAN:,.0f} | growth {g:+.1f}% | plan {pg:+.1f}% | gap {gap:+.1f} pts\n")

# ---- Chart 1: Growth by city against plan ------------------------------------------
s = h.sort_values("growth")
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.barh(s.index, s.growth, color=[PINK if v < g else PURPLE for v in s.growth])
ax.scatter(s.plan_growth, s.index, marker="D", color=DARK, zorder=3, label="plan")
for i, v in enumerate(s.growth):
    ax.text(v + (0.4 if v >= 0 else -0.4), i, f"{v:+.1f}%", va="center", ha="left" if v >= 0 else "right", fontsize=9)
ax.axvline(0, color=DARK, lw=1)
ax.legend(loc="lower right", frameon=False)
ax.set_xlabel(f"Tests, H1 2026 vs H1 2025, % change (denominator: {T25:,.0f} tests in H1 2025)")
ax.set_title(f"Growth by city: network {g:+.1f}% against a plan of {pg:+.0f}%")
plt.tight_layout()
fig.savefig(OUT / "slide4_chart.png", dpi=200)
plt.show()
plt.close(fig)

# ---- Chart 2: Where the gap sits ----------------------------------------------------
s = h.sort_values("gap_pts", ascending=False)
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.barh(s.index, s.gap_pts, color=PINK)
for i, v in enumerate(s.gap_pts):
    ax.text(v - 0.1, i, f"{v:.1f}", va="center", ha="right", fontsize=9)
ax.set_xlabel(f"Points of the gap vs plan, as % of H1 2025 tests ({T25:,.0f})")
ax.set_title(f"The {abs(gap):.1f}-point gap, by city (adds up to {gap:.1f})")
plt.tight_layout()
fig.savefig(OUT / "gap_by_city.png", dpi=200)
plt.show()
plt.close(fig)

# ---- Chart 3: The tree branches ------------------------------------------------------
tn = pd.read_csv(CLEAN / "tree_network.csv", index_col=[0, 1]).xs("Network")
a_, c_ = tn.loc["H1_2025"], tn.loc["H1_2026"]
br = {
    "People": (c_.people / a_.people - 1) * 100,
    "Visits per person": (c_.visits_per_person / a_.visits_per_person - 1) * 100,
    "Tests per visit": (c_.tests_per_visit / a_.tests_per_visit - 1) * 100,
    "Tests (total)": g
}
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(list(br), list(br.values()), color=[PURPLE, PURPLE, PURPLE, DARK])
for i, v in enumerate(br.values()):
    ax.text(i, v + (0.15 if v >= 0 else -0.15), f"{v:+.1f}%", ha="center", va="bottom" if v >= 0 else "top")
ax.axhline(0, color=DARK, lw=1)
ax.set_ylabel("% change, H1 2026 vs H1 2025")
ax.set_title("Which branch of the tree moved")
plt.tight_layout()
fig.savefig(OUT / "tree_branches.png", dpi=200)
plt.show()
plt.close(fig)
weakest = min(list(br)[:3], key=lambda k: br[k])

# ---- Chart 4: Month by month, actual against plan -----------------------------------
m = b.assign(month=b.slot_at.dt.strftime("%Y-%m")).groupby(["city", "month"]).size().rename("n").reset_index()
fig, axes = plt.subplots(2, 3, figsize=(11, 5.5), sharex=True)
for ax, city in zip(axes.flat, sorted(h.index)):
    mm = m[m.city == city].set_index("month").n
    pl = plan[plan.city == city].set_index("month").plan
    mo = [f"2026-0{i}" for i in range(1, 7)]
    ax.plot(range(6), [mm.get(x.replace("2026", "2025"), np.nan) for x in mo], color=GREY, label="2025")
    ax.plot(range(6), [mm.get(x, np.nan) for x in mo], color=PURPLE, lw=2, label="2026")
    ax.plot(range(6), [pl.get(x, np.nan) for x in mo], color=PINK, ls="--", label="plan")
    ax.set_title(city, fontsize=10)
    ax.set_xticks(range(6))
    ax.set_xticklabels(["Jan", "Feb", "Mar", "Apr", "May", "Jun"], fontsize=8)
axes.flat[0].legend(frameon=False, fontsize=8)
fig.suptitle("Tests per month: 2025, 2026 and plan", fontsize=10)
plt.tight_layout()
fig.savefig(OUT / "monthly_by_city.png", dpi=160)
plt.show()
plt.close(fig)

# ---- Chart 5: City x channel growth --------------------------------------------------
cc = pd.read_csv(CLEAN / "city_channel_growth.csv", index_col=[0, 1])["growth_pct"].unstack("channel")
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cc.values, cmap="RdBu", vmin=-max(abs(cc.values.min()), abs(cc.values.max())), vmax=max(abs(cc.values.min()), abs(cc.values.max())))
ax.set_xticks(range(cc.shape[1]))
ax.set_xticklabels(cc.columns)
ax.set_yticks(range(cc.shape[0]))
ax.set_yticklabels(cc.index)
for i in range(cc.shape[0]):
    for j in range(cc.shape[1]):
        ax.text(j, i, f"{cc.values[i, j]:+.1f}%", ha="center", va="center", fontsize=9)
ax.set_title("Growth by city and channel")
plt.tight_layout()
fig.savefig(OUT / "city_channel_heatmap.png", dpi=200)
plt.show()
plt.close(fig)

# ---- Auto-Draft Sentence -------------------------------------------------------------
worst = h.gap_pts.nsmallest(2)
share = worst.sum() / gap * 100 if gap else np.nan
DRAFT = (f"Test volume {'grew' if g >= 0 else 'fell'} {abs(g):.1f}% in H1 2026 against H1 2025 ({T26:,.0f} vs {T25:,.0f} tests, six cities) versus a plan of {pg:+.0f}%; "
         f"{share:.0f}% of the {abs(gap):.1f}-point shortfall sits in {worst.index[0]} and {worst.index[1]}, and the weakest branch is {weakest.lower()}.")
sentence = CLAIM or DRAFT
print("\nDRAFT SENTENCE:\n", DRAFT)
if CAVEAT.startswith("<"):
    print("REMINDER: CAVEAT is still a placeholder.")

# ---- Board Number Excel Export -------------------------------------------------------
wb = Workbook()
ws = wb.active
ws.title = "Board number"
rows = [
    (f"Kalpa Health | Brief {BRIEF} | Group {GROUP}", ""),
    ("", ""),
    ("The number", f"{g:+.1f}%"),
    ("What it counts", "tests (booking-test rows) after cleaning rules"),
    ("Numerator", int(T26)),
    ("Denominator", int(T25)),
    ("Period", "H1 2026 against H1 2025, six cities"),
    ("Compared with", f"plan: {pg:+.0f}% ({gap:+.1f} point gap)"),
    ("The sentence", sentence),
    ("The caveat", CAVEAT),
    ("Formula check", "=ROUND((B5/B6-1)*100,1)")
]
for r in rows:
    ws.append(list(r))
ws["A1"].font = Font(bold=True, size=14)
ws["B3"].font = Font(bold=True, size=24)
ws.column_dimensions["A"].width = 18
ws.column_dimensions["B"].width = 90
for row in ws.iter_rows(min_row=3, max_row=11):
    row[1].alignment = Alignment(wrap_text=True, vertical="top")
wb.save(OUT / "board_number.xlsx")
print("\nSaved all charts and board_number.xlsx to the 'slides/' directory!")

/tmp/ipykernel_13292/2432044575.py:23: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  b = pd.read_csv(CLEAN / "bookings_clean.csv.gz", parse_dates=["slot_at"])


NETWORK  H1 2025 435,430 | H1 2026 457,230 | plan 515,740 | growth +5.0% | plan +18.4% | gap -13.4 pts


DRAFT SENTENCE:
 Test volume grew 5.0% in H1 2026 against H1 2025 (457,230 vs 435,430 tests, six cities) versus a plan of +18%; 44% of the 13.4-point shortfall sits in Chennai and Bengaluru, and the weakest branch is visits per person.
REMINDER: CAVEAT is still a placeholder.

Saved all charts and board_number.xlsx to the 'slides/' directory!


In [36]:
"""Step 7: the charts for the slides, the board number in Excel, and a draft of the one sentence.
Needs Steps 5 and 6 run.  Run from repo root:  python src/step7_charts_board.py"""
import pathlib, numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment

# ---- EDIT THESE after you have seen the numbers ------------------------------------
BRIEF, GROUP = "A", "<N>"
CLAIM  = None        # your final sentence, in quotes. Leave None to use the auto-draft.
CAVEAT = "<the one thing that would change the claim, and its size>"
# ------------------------------------------------------------------------------------
RAW, CLEAN, OUT = pathlib.Path(workbench_path), pathlib.Path(workbench_path), pathlib.Path("slides"); OUT.mkdir(exist_ok=True)
PURPLE, PINK, DARK, GREY = "#4E38AB", "#D63A6A", "#1A1440", "#9A96B5"

b = pd.read_csv(CLEAN / "bookings_clean.csv.gz", parse_dates=["slot_at"])
b["half"] = np.where(b.slot_at.between("2025-01-01", "2025-06-30 23:59:59"), "H1_2025", np.where(b.slot_at.between("2026-01-01", "2026-06-30 23:59:59"), "H1_2026", "other"))
plan = pd.read_csv(workbench_path / "plan_2026_H1.csv").melt(id_vars=["city", "metric"], var_name="month", value_name="plan")
plan["city"] = plan.city.replace({"Bangalore": "Bengaluru"})

# ---- numbers every chart uses -------------------------------------------------------
h = b[b.half != "other"].groupby(["city", "half"]).size().unstack("half").fillna(0)
h["plan"] = plan.groupby("city").plan.sum()
T25, T26, TPLAN = h.H1_2025.sum(), h.H1_2026.sum(), h.plan.sum()
h["growth"] = (h.H1_2026 / h.H1_2025 - 1) * 100
h["plan_growth"] = (h.plan / h.H1_2025 - 1) * 100
h["gap_pts"] = (h.H1_2026 - h.plan) / T25 * 100
g, pg, gap = (T26 / T25 - 1) * 100, (TPLAN / T25 - 1) * 100, h.gap_pts.sum()
print(f"NETWORK  H1 2025 {T25:,.0f} | H1 2026 {T26:,.0f} | plan {TPLAN:,.0f} | growth {g:+.1f}% | plan {pg:+.1f}% | gap {gap:+.1f} pts")

# ---- chart 1 (the slide-4 chart): growth by city against plan -----------------------
s = h.sort_values("growth"); fig, ax = plt.subplots(figsize=(8, 4.2))
ax.barh(s.index, s.growth, color=[PINK if v < g else PURPLE for v in s.growth])
ax.scatter(s.plan_growth, s.index, marker="D", color=DARK, zorder=3, label="plan")
for i, v in enumerate(s.growth): ax.text(v + (0.4 if v >= 0 else -0.4), i, f"{v:+.1f}%", va="center", ha="left" if v >= 0 else "right", fontsize=9)
ax.axvline(0, color=DARK, lw=1); ax.legend(loc="lower right", frameon=False)
ax.set_xlabel(f"Tests, H1 2026 vs H1 2025, % change (denominator: {T25:,.0f} tests in H1 2025)")
ax.set_title(f"Growth by city: network {g:+.1f}% against a plan of {pg:+.0f}%"); plt.tight_layout(); fig.savefig(OUT / "slide4_chart.png", dpi=200); plt.close()

# ---- chart 2: where the gap sits (adds to the network gap) --------------------------
s = h.sort_values("gap_pts", ascending=False); fig, ax = plt.subplots(figsize=(8, 4.2))
ax.barh(s.index, s.gap_pts, color=PINK)
for i, v in enumerate(s.gap_pts): ax.text(v - 0.1, i, f"{v:.1f}", va="center", ha="right", fontsize=9)
ax.set_xlabel(f"Points of the gap vs plan, as % of H1 2025 tests ({T25:,.0f})"); ax.set_title(f"The {abs(gap):.1f}-point gap, by city (adds up to {gap:.1f})")
plt.tight_layout(); fig.savefig(OUT / "gap_by_city.png", dpi=200); plt.close()

# ---- chart 3: the tree branches ------------------------------------------------------
tn = pd.read_csv(CLEAN / "tree_network.csv", index_col=[0, 1]).xs("Network")
a_, c_ = tn.loc["H1_2025"], tn.loc["H1_2026"]
br = {"People": (c_.people / a_.people - 1) * 100, "Visits per person": (c_.visits_per_person / a_.visits_per_person - 1) * 100,
      "Tests per visit": (c_.tests_per_visit / a_.tests_per_visit - 1) * 100, "Tests (total)": g}
fig, ax = plt.subplots(figsize=(7, 4)); ax.bar(list(br), list(br.values()), color=[PURPLE, PURPLE, PURPLE, DARK])
for i, v in enumerate(br.values()): ax.text(i, v + (0.15 if v >= 0 else -0.15), f"{v:+.1f}%", ha="center", va="bottom" if v >= 0 else "top")
ax.axhline(0, color=DARK, lw=1); ax.set_ylabel("% change, H1 2026 vs H1 2025"); ax.set_title("Which branch of the tree moved"); plt.tight_layout()
fig.savefig(OUT / "tree_branches.png", dpi=200); plt.close()
weakest = min(list(br)[:3], key=lambda k: br[k])

# ---- chart 4: month by month, actual against plan, one panel per city ----------------
m = b.assign(month=b.slot_at.dt.strftime("%Y-%m")).groupby(["city", "month"]).size().rename("n").reset_index()
fig, axes = plt.subplots(2, 3, figsize=(11, 5.5), sharex=True)
for ax, city in zip(axes.flat, sorted(h.index)):
    mm = m[m.city == city].set_index("month").n; pl = plan[plan.city == city].set_index("month").plan
    mo = [f"2026-0{i}" for i in range(1, 7)]
    ax.plot(range(6), [mm.get(x.replace("2026", "2025"), np.nan) for x in mo], color=GREY, label="2025")
    ax.plot(range(6), [mm.get(x, np.nan) for x in mo], color=PURPLE, lw=2, label="2026")
    ax.plot(range(6), [pl.get(x, np.nan) for x in mo], color=PINK, ls="--", label="plan")
    ax.set_title(city, fontsize=10); ax.set_xticks(range(6)); ax.set_xticklabels(["Jan", "Feb", "Mar", "Apr", "May", "Jun"], fontsize=8)
axes.flat[0].legend(frameon=False, fontsize=8); fig.suptitle("Tests per month: 2025, 2026 and plan (a drop starting in April in Chennai or Jaipur = system change, check it)", fontsize=10)
plt.tight_layout(); fig.savefig(OUT / "monthly_by_city.png", dpi=160); plt.close()

# ---- chart 5: city x channel growth --------------------------------------------------
cc = pd.read_csv(CLEAN / "city_channel_growth.csv", index_col=[0, 1])["growth_pct"].unstack("channel")
fig, ax = plt.subplots(figsize=(5, 4)); im = ax.imshow(cc.values, cmap="RdBu", vmin=-max(abs(cc.values.min()), abs(cc.values.max())), vmax=max(abs(cc.values.min()), abs(cc.values.max())))
ax.set_xticks(range(cc.shape[1])); ax.set_xticklabels(cc.columns); ax.set_yticks(range(cc.shape[0])); ax.set_yticklabels(cc.index)
for i in range(cc.shape[0]):
    for j in range(cc.shape[1]): ax.text(j, i, f"{cc.values[i, j]:+.1f}%", ha="center", va="center", fontsize=9)
ax.set_title("Growth by city and channel"); plt.tight_layout(); fig.savefig(OUT / "city_channel_heatmap.png", dpi=200); plt.close()

# ---- the sentence: an auto-draft to edit, never to submit untouched -------------------
worst = h.gap_pts.nsmallest(2); share = worst.sum() / gap * 100 if gap else np.nan
DRAFT = (f"Test volume {'grew' if g >= 0 else 'fell'} {abs(g):.1f}% in H1 2026 against H1 2025 ({T26:,.0f} vs {T25:,.0f} tests, six cities) versus a plan of {pg:+.0f}%; "
         f"{share:.0f}% of the {abs(gap):.0f}-point shortfall sits in {worst.index[0]} and {worst.index[1]}, and the weakest branch is {weakest.lower()}.")
sentence = CLAIM or DRAFT
print("\nDRAFT SENTENCE (edit it, check every number, then paste into CLAIM at the top):\n ", DRAFT)
if CAVEAT.startswith("<"): print("REMINDER: CAVEAT is still a placeholder.")

# ---- the board number, in Excel, with a live formula check -----------------------------
wb = Workbook(); ws = wb.active; ws.title = "Board number"
rows = [(f"Kalpa Health | Brief {BRIEF} | Group {GROUP}", ""), ("", ""), ("The number", f"{g:+.1f}%"), ("What it counts", "tests (booking-test rows) after the cleaning rules in docs/decisions-log.md"),
        ("Numerator", int(T26)), ("Denominator", int(T25)), ("Period", "H1 2026 against H1 2025, six cities"), ("Compared with", f"plan: {pg:+.0f}%  ({gap:+.1f} point gap)"),
        ("The sentence", sentence), ("The caveat", CAVEAT), ("Formula check", "=ROUND((B5/B6-1)*100,1)")]
for r in rows: ws.append(list(r))
ws["A1"].font = Font(bold=True, size=14); ws["B3"].font = Font(bold=True, size=24)
ws.column_dimensions["A"].width = 18; ws.column_dimensions["B"].width = 90
for row in ws.iter_rows(min_row=3, max_row=11): row[1].alignment = Alignment(wrap_text=True, vertical="top")
wb.save(OUT / "board_number.xlsx"); print("saved slides/: slide4_chart, gap_by_city, tree_branches, monthly_by_city, city_channel_heatmap (png) and board_number.xlsx")

/tmp/ipykernel_13292/1060936637.py:17: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  b = pd.read_csv(CLEAN / "bookings_clean.csv.gz", parse_dates=["slot_at"])


NETWORK  H1 2025 435,430 | H1 2026 457,230 | plan 515,740 | growth +5.0% | plan +18.4% | gap -13.4 pts

DRAFT SENTENCE (edit it, check every number, then paste into CLAIM at the top):
  Test volume grew 5.0% in H1 2026 against H1 2025 (457,230 vs 435,430 tests, six cities) versus a plan of +18%; 44% of the 13-point shortfall sits in Chennai and Bengaluru, and the weakest branch is visits per person.
REMINDER: CAVEAT is still a placeholder.
saved slides/: slide4_chart, gap_by_city, tree_branches, monthly_by_city, city_channel_heatmap (png) and board_number.xlsx


In [40]:
"""Step 8: try to break our own claim.
 (1) sensitivity table: the headline under each alternative cleaning decision
 (2) recording-problem check: do bookings and the independent collections app tell the same story, city by city?
 (3) what-if: what the weakest branch would have to do to hit the plan
Run from repo root:  python src/step8_stress_test.py"""
import pathlib, numpy as np, pandas as pd

COUNT_RULE = "A"                                   # same value as in step5_clean.py
RAW, CLEAN = pathlib.Path(workbench_path), pathlib.Path(workbench_path)
H = {"H1_2025": ("2025-01-01", "2025-06-30 23:59:59"), "H1_2026": ("2026-01-01", "2026-06-30 23:59:59")}
CODE2CITY = {"AHM": "Ahmedabad", "BLR": "Bengaluru", "CHE": "Chennai", "HYD": "Hyderabad", "JAI": "Jaipur", "PUN": "Pune"}

# # b = pd.read_csv(CLEAN / "bookings_all.csv.gz", parse_dates=["slot_at"])
# files = sorted((RAW / "collections").glob("*.csv.gz"))
# coll = pd.concat([pd.read_csv(p, dtype=str) for p in files], ignore_index=True)
# coll["booking_id"] = coll.booking_id.str.strip(); coll = coll.drop_duplicates().drop_duplicates("booking_id")
# b = b.merge(coll[["booking_id", "status"]].rename(columns={"status": "coll_status"}), on="booking_id", how="left", validate="many_to_one")
# b["coll_status"] = b.coll_status.fillna("no_collection_row")

def prep(keep_resch=False, keep_empty=False, rule=COUNT_RULE, drop_blank_code=False, cities=None):
    d = b
    if not keep_resch: d = d[~d.status_at_export.eq("rescheduled").fillna(False).astype(bool)]
    if not keep_empty: d = d[~((d.system == "nova") & d.test_code.isna() & d.test_price.isna())]
    if rule == "B": d = d[d.coll_status == "collected"]
    if rule == "C": d = d[d.coll_status.isin(["collected", "no_show"])]
    if drop_blank_code: d = d[d.test_code.notna()]
    if cities: d = d[d.city.isin(cities)]
    return d

def growth(d, unit="tests"):
    f = (lambda x: len(x)) if unit == "tests" else (lambda x: x.booking_id.nunique())
    a, c = f(d[d.slot_at.between(*H["H1_2025"])]), f(d[d.slot_at.between(*H["H1_2026"])])
    return a, c, ((c / a - 1) * 100 if a else np.nan)

# ---- 1. sensitivity table ----------------------------------------------------------------
four = ["Ahmedabad", "Bengaluru", "Hyderabad", "Pune"]
variants = {
    "BASELINE (our decisions)":                 (prep(), "tests"),
    "keep nova rescheduled originals":          (prep(keep_resch=True), "tests"),
    "keep nova empty-test placeholders":        (prep(keep_empty=True), "tests"),
    "count rule A: all rows":                   (prep(rule="A"), "tests"),
    "count rule B: collected only":             (prep(rule="B"), "tests"),
    "count rule C: collected + no-show":        (prep(rule="C"), "tests"),
    "drop rows with blank test_code":           (prep(drop_blank_code=True), "tests"),
    "only the 4 cities that never changed system": (prep(cities=four), "tests"),
    "count visits (bookings), not tests":       (prep(), "bookings"),
}
rows = []
for name, (d, unit) in variants.items():
    a, c, g = growth(d, unit); rows.append({"variant": name, "H1_2025": a, "H1_2026": c, "growth_pct": round(g, 2)})
S = pd.DataFrame(rows); S["moved_vs_baseline_pts"] = (S.growth_pct - S.growth_pct.iloc[0]).round(2)
pd.set_option("display.width", 200)
print("1. SENSITIVITY: the headline under each alternative decision\n", S.to_string(index=False))
S.to_csv(CLEAN / "sensitivity_table.csv", index=False)
print("   -> the three rows with the largest |moved_vs_baseline_pts| go on slide 5 / the sensitivity stretch")

# ---- 2. recording-problem check ----------------------------------------------------------
# Collections come from the phlebotomy app, which did NOT change in April. Bookings come from the system that did.
base = prep(); base = base.assign(q=base.slot_at.dt.to_period("Q").astype(str))
bk_q = base.groupby(["city", "q"]).booking_id.nunique().unstack("q")
cl = coll.assign(city=coll.clinic_id.str[:3].map(CODE2CITY), q=pd.to_datetime(coll.collected_at, errors="coerce").dt.to_period("Q").astype(str))
cl_q = cl[cl.status == "collected"].groupby(["city", "q"]).size().unstack("q")
def yoy(t, q26, q25): return ((t[q26] / t[q25] - 1) * 100) if q26 in t and q25 in t else pd.Series(np.nan, index=t.index)
R = pd.DataFrame({"bookings_Q1": yoy(bk_q, "2026Q1", "2025Q1"), "bookings_Q2": yoy(bk_q, "2026Q2", "2025Q2"),
                  "collected_Q1": yoy(cl_q, "2026Q1", "2025Q1"), "collected_Q2": yoy(cl_q, "2026Q2", "2025Q2")}).round(1)
R["bookings_change_Q1_to_Q2"] = (R.bookings_Q2 - R.bookings_Q1).round(1)
R["collected_change_Q1_to_Q2"] = (R.collected_Q2 - R.collected_Q1).round(1)
print("\n2. RECORDING CHECK: year-on-year growth %, by city and quarter (bookings = changed system, collected = independent app)\n", R.to_string())
print("   Read it: if a city's BOOKINGS growth drops from Q1 to Q2 but its COLLECTED growth does not, the drop is a counting artefact.")
R.to_csv(CLEAN / "recording_check.csv")

# ---- 3. what-if ---------------------------------------------------------------------------
tn = pd.read_csv(CLEAN / "tree_network.csv", index_col=[0, 1]).xs("Network")
a_, c_ = tn.loc["H1_2025"], tn.loc["H1_2026"]
f = {k: c_[k] / a_[k] for k in ["people", "visits_per_person", "tests_per_visit"]}
weak = min(f, key=f.get); plan_total = pd.read_csv(RAW / "plan" / "plan_2026_H1.csv").drop(columns=["city", "metric"]).sum().sum()
need = (plan_total / a_.tests) / np.prod([v for k, v in f.items() if k != weak])
print(f"\n3. WHAT-IF: weakest branch = {weak}: grew {(f[weak]-1)*100:+.1f}%. To reach the plan ({plan_total:,.0f} tests) with the other two branches as observed,")
print(f"   it would have needed {(need-1)*100:+.1f}%, which is {(plan_total - c_.tests):,.0f} more tests than the {c_.tests:,.0f} we had.")
print("   ASSUMPTION to state on the slide: the other two branches stay exactly as observed.")

AttributeError: 'DataFrame' object has no attribute 'coll_status'